# URI, URL, and URN - JavaScript

All 10 JavaScript examples from [docs/uri.md](https://platob.github.io/yggdryl/uri/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install yggdryl
```

In [ ]:
const assert = require('node:assert/strict')
const { Uri } = require('yggdryl')

const uri = Uri.from('HTTPS://example.test/archive/report.tar.gz?q=1#summary')

assert.equal(uri.toString(), 'https://example.test/archive/report.tar.gz?q=1#summary')
assert.equal(uri.scheme, 'https')
assert.equal(uri.authority, 'example.test')
assert.equal(uri.path, '/archive/report.tar.gz')
assert.equal(uri.query, 'q=1')
assert.equal(uri.fragment, 'summary')
assert.equal(uri.fileName, 'report.tar.gz')

## Canonical on arrival

In [ ]:
const assert = require('node:assert/strict')
const { Uri } = require('yggdryl')

const uri = Uri.from('HTTPS://example.test/caf%c3%a9.csv')
assert.equal(uri.toString(), 'https://example.test/caf%C3%A9.csv')

const windows = Uri.from('file:///C:\\Users\\Ada\\report.parquet')
assert.equal(windows.toString(), 'file:///C:/Users/Ada/report.parquet')

assert.equal(Uri.from('/var/lib/data.arrow').toString(), 'file:///var/lib/data.arrow')
assert.equal(Uri.from('data/ticks.csv').toString(), 'file:data/ticks.csv')

const stamped = Uri.from('/data/2026-08-16T00:00:00/part.parquet')
assert.equal(stamped.scheme, 'file')

assert.ok(Uri.from(uri.toString()).equals(uri))

## Credentials and S3 locations

In [ ]:
const assert = require('node:assert/strict')
const { Uri } = require('yggdryl')

const secured = Uri.from('https://user:pass:word@example.com/data')
assert.deepEqual(
  [secured.user, secured.password, secured.hostname],
  ['user', 'pass:word', 'example.com'],
)

const s3 = Uri.from('s3://trades.s3.eu-west-3.amazonaws.com/part.parquet')
assert.deepEqual([s3.bucket, s3.region], ['trades', 'eu-west-3'])

## Path segments

In [ ]:
const assert = require('node:assert/strict')
const { Uri } = require('yggdryl')

const uri = Uri.from('https://example.test/archive/2026/report.tar.gz')

assert.deepEqual(uri.pathSegments, ['archive', '2026', 'report.tar.gz'])
assert.equal(uri.length, 3)
assert.equal(uri.at(0), 'archive')
assert.equal(uri.at(-1), 'report.tar.gz')
assert.deepEqual([...uri], uri.pathSegments)

## Compound filenames

In [ ]:
const assert = require('node:assert/strict')
const { Uri } = require('yggdryl')

const uri = Uri.from('https://example.test/archive/report.tar.gz?q=1#part')

assert.equal(uri.fileName, 'report.tar.gz')
assert.equal(uri.stem, 'report.tar')
assert.equal(uri.extension, 'gz')
assert.deepEqual(uri.extensions, ['tar', 'gz'])

uri.setStem('renamed')
assert.equal(uri.toString(), 'https://example.test/archive/renamed.gz?q=1#part')
uri.setExtensions(['csv', 'gz'])
assert.equal(uri.toString(), 'https://example.test/archive/renamed.csv.gz?q=1#part')
assert.equal(uri.removeExtension(), true)
assert.equal(uri.clearExtensions(), true)
assert.equal(uri.toString(), 'https://example.test/archive/renamed?q=1#part')

const unchanged = uri.toString()
assert.throws(() => uri.setFileName('bad/name'))
assert.equal(uri.toString(), unchanged)

## The media type is in the name

In [ ]:
const assert = require('node:assert/strict')
const { MediaType, MimeType, Uri } = require('yggdryl')

const uri = Uri.from('https://example.test/report.csv.gz.zst?q=1#part')

assert.ok(uri.mimeType.equals(MimeType.from('application/zstd')))
assert.ok(uri.mediaType.base.equals(MimeType.from('text/csv')))
assert.deepEqual(
  uri.mediaType.encodings.map((value) => value.toString()),
  ['application/gzip', 'application/zstd'],
)

uri.setMimeType('application/json')
assert.equal(uri.toString(), 'https://example.test/report.csv.gz.json?q=1#part')

uri.setMediaType(MediaType.fromParts('text/csv', ['application/gzip', 'application/zstd']))
assert.equal(uri.toString(), 'https://example.test/report.csv.gz.zst?q=1#part')

const unchanged = uri.toString()
assert.throws(() => uri.setMimeType('application/vnd.example'), /preferred filename extension/)
assert.equal(uri.toString(), unchanged)

## URL and URN

In [ ]:
const assert = require('node:assert/strict')
const { Uri, Url, Urn } = require('yggdryl')

const uri = Uri.from('https://example.test/a/data.json?raw=true')
const url = Url.from(uri)
assert.equal(url.authority, 'example.test')
assert.ok(Uri.from(url).equals(uri))

const urn = Urn.from('URN:ISBN:9780131103627')
assert.equal(urn.toString(), 'urn:isbn:9780131103627')
assert.equal(urn.namespace, 'isbn')
assert.equal(urn.namespaceSpecific, '9780131103627')
assert.equal(urn.authority, '')

assert.throws(() => urn.intoUri().intoUrl())
assert.throws(() => Urn.fromUri(uri))
assert.throws(() => Url.fromString('mailto:user@example.test'))

## Platform paths

In [ ]:
const assert = require('node:assert/strict')
const { Uri, Url } = require('yggdryl')

const uri = Uri.fromPath('C:\\Users\\Ada Lovelace\\report.parquet')
assert.equal(uri.toString(), 'file:///C:/Users/Ada%20Lovelace/report.parquet')
assert.equal(uri.authority, '')
assert.equal(uri.fileName, 'report.parquet')

assert.equal(uri.intoPath(), 'C:/Users/Ada Lovelace/report.parquet')
assert.ok(Uri.fromPath(uri.intoPath()).equals(uri))

const unc = Uri.fromPath('\\\\server\\share\\prices\\ticks.csv')
assert.equal(unc.toString(), 'file://server/share/prices/ticks.csv')
assert.equal(unc.authority, 'server')
assert.equal(unc.intoPath(), '//server/share/prices/ticks.csv')

assert.throws(() => Url.fromString('https://example.test/data.csv').intoPath())

## Walking the path

In [ ]:
const assert = require('node:assert/strict')
const { Url } = require('yggdryl')

const url = Url.from('https://example.test/a/b/c?q=1#frag')

// `joinpath` is variadic the way `path.join` is; everything else survives.
assert.equal(url.joinpath('d').toString(), 'https://example.test/a/b/c/d?q=1#frag')
assert.equal(url.joinpath('d', 'e').toString(), 'https://example.test/a/b/c/d/e?q=1#frag')

// `parts` is the sequence of names the path actually addresses.
assert.deepEqual(url.parts, ['a', 'b', 'c'])

// `parents` climbs to the root and never yields the value itself.
assert.deepEqual(url.parents.map((parent) => parent.path), ['/a/b', '/a', '/'])
assert.equal(url.parent.path, '/a/b')

## Patterns and partitions

In [ ]:
const assert = require('node:assert/strict')
const { Url } = require('yggdryl')

const pattern = Url.from('file:///lake/trades/year=2024/**/*.parquet')
assert.ok(pattern.isGlob())

// Matching follows the `.gitignore` rule.
const part = Url.from('file:///lake/trades/year=2024/month=01/part-0.parquet')
assert.ok(part.match('*.parquet'))
assert.ok(part.match('lake/**/part-?.parquet'))
assert.ok(!part.match('lake/*.parquet'))

// The directory names are the partition columns.
assert.equal(part.partition('month'), '01')
assert.deepEqual(part.partitions, [
  { column: 'year', value: '2024' },
  { column: 'month', value: '01' },
])
assert.equal(
  part.relativeTo('file:///lake/trades'),
  'year=2024/month=01/part-0.parquet',
)